# 02 Baseline Strategies: Static Band + TWAP/VWAP

This notebook is the first empirical hedging layer after the simulated executed-trade tape. It follows the updated project plan in `docs/`:

- Convert executed option trades into signed delta arrivals.
- Backtest zero-hedge and full-hedge benchmarks.
- Backtest static no-trade bands around zero delta.
- Execute band-triggered hedge orders using TWAP and VWAP schedules.
- Measure execution cost, residual delta risk, and the combined objective.

Primary input: `data/simulated/simulated_executed_trades.csv`.

Primary output: `outputs/tables/baseline_strategy_results.csv`.

## 1. Setup

In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data'
SIM_DIR = DATA_DIR / 'simulated'
PROCESSED_DIR = DATA_DIR / 'processed'
OUTPUT_TABLE_DIR = PROJECT_ROOT / 'outputs' / 'tables'
OUTPUT_FIGURE_DIR = PROJECT_ROOT / 'outputs' / 'figures'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

EXECUTED_TRADES_PATH = SIM_DIR / 'simulated_executed_trades.csv'
MARKET_INPUTS_PATH = PROCESSED_DIR / 'market_inputs_2026.csv'

OUTPUT_DELTA_ARRIVALS = PROCESSED_DIR / 'executed_trade_delta_arrivals.csv'
OUTPUT_HEDGE_EVENTS = PROCESSED_DIR / 'baseline_hedge_events.csv'
OUTPUT_RESULTS = OUTPUT_TABLE_DIR / 'baseline_strategy_results.csv'

ES_MULTIPLIER = 50
BAR_MINUTES = 5
BAR_YEAR_FRACTION = BAR_MINUTES / (252 * 6.5 * 60)

BAND_GRID = [25_000, 50_000, 100_000, 200_000]
HORIZON_GRID_MINUTES = [15, 30, 60]
EXECUTION_METHODS = ['twap', 'vwap']

SPREAD_BPS = 1.0
IMPACT_ALPHA = 0.05
IMPACT_BETA = 0.5
LAMBDA_RISK = 1e-8

print(PROJECT_ROOT)

## 2. Load Executed Trades and Market Inputs

In [ ]:
def load_executed_trades(path: Path) -> pd.DataFrame:
    trades = pd.read_csv(path)
    trades['timestamp_utc'] = pd.to_datetime(trades['timestamp'], utc=True)
    trades['timestamp_et'] = trades['timestamp_utc'].dt.tz_convert('America/New_York').dt.tz_localize(None)
    trades['trade_date'] = trades['timestamp_et'].dt.normalize()
    trades['expiry_date'] = pd.to_datetime(trades['expiry_date'])
    trades['side'] = trades['side'].str.upper()
    trades['opt_type'] = trades['opt_type'].str.upper()
    return trades.sort_values('timestamp_et').reset_index(drop=True)


def load_market_inputs(path: Path) -> pd.DataFrame:
    market = pd.read_csv(path)
    market['timestamp_et'] = pd.to_datetime(market['timestamp_et'])
    market['trade_date'] = pd.to_datetime(market['trade_date'])
    return market.sort_values('timestamp_et').reset_index(drop=True)


trades_raw = load_executed_trades(EXECUTED_TRADES_PATH)
market = load_market_inputs(MARKET_INPUTS_PATH)

print(f'Executed trades: {len(trades_raw):,}')
print(f'Market bars:     {len(market):,}')
trades_raw.head()

## 3. Align Trades to 5-Minute Market Bars

The execution tape is timestamped at trade time. The backtest works on the same 5-minute grid as the market inputs, so each trade is assigned to the latest available market bar.

In [ ]:
def align_trades_to_market(trades: pd.DataFrame, market: pd.DataFrame) -> pd.DataFrame:
    aligned = pd.merge_asof(
        trades.sort_values('timestamp_et'),
        market.sort_values('timestamp_et'),
        on='timestamp_et',
        direction='backward',
        tolerance=pd.Timedelta(minutes=10),
        suffixes=('', '_mkt'),
    )
    missing = aligned['es1_price'].isna().sum()
    if missing:
        print(f'Dropping {missing:,} trades that could not be aligned to a market bar.')
        aligned = aligned.dropna(subset=['es1_price']).copy()
    aligned['bar_time'] = aligned['timestamp_et'].dt.floor(f'{BAR_MINUTES}min')
    return aligned.reset_index(drop=True)


trades_aligned = align_trades_to_market(trades_raw, market)
trades_aligned[['trade_id', 'timestamp_et', 'bar_time', 'symbol', 'opt_type', 'strike', 'quantity', 'es1_price', 'bvol_1m']].head()

## 4. Convert Option Executions to Delta Arrivals

The executed-trade tape is ES option flow, so the framework uses a Black-76-style futures-option delta. This cell creates the signed book-delta arrival from each option execution.

In [ ]:
def norm_cdf(x: float) -> float:
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def black76_delta(futures_price: float, strike: float, years: float, vol: float, option_type: str, rate: float = 0.0) -> float:
    years = max(float(years), 1 / 365)
    vol = max(float(vol), 0.01)
    futures_price = max(float(futures_price), 0.01)
    strike = max(float(strike), 0.01)
    discount = math.exp(-float(rate) * years)
    d1 = (math.log(futures_price / strike) + 0.5 * vol**2 * years) / (vol * math.sqrt(years))
    if option_type == 'C':
        return discount * norm_cdf(d1)
    return -discount * norm_cdf(-d1)


def add_delta_arrivals(trades: pd.DataFrame) -> pd.DataFrame:
    out = trades.copy()
    out['dte'] = (out['expiry_date'] - out['trade_date']).dt.days.clip(lower=1)
    out['years_to_expiry'] = out['dte'] / 365.0
    out['delta'] = [
        black76_delta(row.es1_price, row.strike, row.years_to_expiry, row.bvol_1m, row.opt_type, row.sofr_1m_rate)
        for row in out.itertuples(index=False)
    ]
    out['side_sign'] = np.where(out['side'].eq('BUY'), 1.0, -1.0)
    out['signed_delta_contracts'] = out['side_sign'] * out['delta'] * out['quantity'] * ES_MULTIPLIER
    out['signed_delta_notional'] = out['signed_delta_contracts'] * out['es1_price']
    return out


delta_arrivals = add_delta_arrivals(trades_aligned)
delta_arrivals.to_csv(OUTPUT_DELTA_ARRIVALS, index=False)

delta_arrivals[['trade_id', 'timestamp_et', 'symbol', 'side', 'quantity', 'delta', 'signed_delta_contracts', 'signed_delta_notional']].head()

## 5. Build the Backtest Grid

The event grid combines market bars with delta arrivals. The strategy simulator will update book delta at each bar, apply scheduled child hedge orders, then decide whether a new hedge schedule should replace the current one.

In [ ]:
def build_delta_bar_grid(delta_arrivals: pd.DataFrame, market: pd.DataFrame) -> pd.DataFrame:
    arrivals = (
        delta_arrivals.groupby('bar_time', as_index=False)
        .agg(
            option_delta_arrival=('signed_delta_contracts', 'sum'),
            option_delta_notional=('signed_delta_notional', 'sum'),
            n_option_trades=('trade_id', 'count'),
        )
    )
    grid = market.rename(columns={'timestamp_et': 'bar_time'}).copy()
    grid = grid.merge(arrivals, on='bar_time', how='left')
    grid[['option_delta_arrival', 'option_delta_notional', 'n_option_trades']] = grid[
        ['option_delta_arrival', 'option_delta_notional', 'n_option_trades']
    ].fillna(0.0)
    grid['trade_date'] = pd.to_datetime(grid['trade_date'])
    return grid.sort_values('bar_time').reset_index(drop=True)


bar_grid = build_delta_bar_grid(delta_arrivals, market)
bar_grid[['bar_time', 'trade_date', 'es1_price', 'es1_volume_5m', 'option_delta_arrival', 'n_option_trades']].head()

## 6. Execution Cost and Scheduling Helpers

In [ ]:
def estimate_child_order_cost(delta_contracts: float, price: float, adv: float) -> tuple[float, float, float]:
    """Return total cost, spread cost, and impact cost for an ES child hedge order."""
    qty = abs(float(delta_contracts))
    price = float(price)
    adv = max(float(adv), 1.0)
    half_spread = price * (SPREAD_BPS / 10_000) / 2
    spread_cost = qty * half_spread
    participation = qty / adv
    impact_cost = IMPACT_ALPHA * price * qty * (participation ** IMPACT_BETA)
    return spread_cost + impact_cost, spread_cost, impact_cost


def make_child_schedule(parent_delta: float, start_idx: int, day_grid: pd.DataFrame, horizon_minutes: int, method: str) -> pd.DataFrame:
    n_child = max(int(horizon_minutes / BAR_MINUTES), 1)
    idx = list(range(start_idx, min(start_idx + n_child, len(day_grid))))
    if not idx:
        return pd.DataFrame(columns=['idx', 'hedge_delta'])

    if method == 'twap':
        weights = np.ones(len(idx), dtype=float) / len(idx)
    elif method == 'vwap':
        volume = day_grid.loc[idx, 'es1_volume_5m'].clip(lower=0).fillna(0).to_numpy(dtype=float)
        weights = volume / volume.sum() if volume.sum() > 0 else np.ones(len(idx), dtype=float) / len(idx)
    else:
        raise ValueError(f'Unknown execution method: {method}')

    return pd.DataFrame({'idx': idx, 'hedge_delta': parent_delta * weights})

## 7. Strategy Simulator Framework

In [ ]:
def simulate_static_band_day(day_grid: pd.DataFrame, band: float, method: str, horizon_minutes: int) -> tuple[dict, pd.DataFrame]:
    book_delta = 0.0
    active_schedule = pd.DataFrame(columns=['idx', 'hedge_delta'])
    hedge_events = []
    risk_cost = 0.0
    exec_cost = 0.0
    spread_cost = 0.0
    impact_cost = 0.0
    total_abs_hedge = 0.0
    triggers = 0

    day_grid = day_grid.reset_index(drop=True).copy()

    for i, row in day_grid.iterrows():
        book_delta += float(row.option_delta_arrival)

        scheduled = active_schedule.loc[active_schedule['idx'].eq(i), 'hedge_delta']
        hedge_delta = float(scheduled.sum()) if not scheduled.empty else 0.0
        if hedge_delta != 0.0:
            cost, spread, impact = estimate_child_order_cost(hedge_delta, row.es1_price, row.es1_adv_20d)
            exec_cost += cost
            spread_cost += spread
            impact_cost += impact
            total_abs_hedge += abs(hedge_delta)
            book_delta += hedge_delta
            hedge_events.append({
                'bar_time': row.bar_time,
                'event_type': 'child_order',
                'band': band,
                'method': method,
                'horizon_minutes': horizon_minutes,
                'hedge_delta': hedge_delta,
                'book_delta_after': book_delta,
                'exec_cost': cost,
                'spread_cost': spread,
                'impact_cost': impact,
            })

        risk_cost += (book_delta ** 2) * (float(row.bvol_1m) ** 2) * BAR_YEAR_FRACTION

        if abs(book_delta) > band:
            triggers += 1
            parent_order = -book_delta
            active_schedule = make_child_schedule(parent_order, i + 1, day_grid, horizon_minutes, method)
            hedge_events.append({
                'bar_time': row.bar_time,
                'event_type': 'trigger',
                'band': band,
                'method': method,
                'horizon_minutes': horizon_minutes,
                'hedge_delta': parent_order,
                'book_delta_after': book_delta,
                'exec_cost': 0.0,
                'spread_cost': 0.0,
                'impact_cost': 0.0,
            })

    close_delta = -book_delta
    if close_delta != 0.0:
        row = day_grid.iloc[-1]
        cost, spread, impact = estimate_child_order_cost(close_delta, row.es1_price, row.es1_adv_20d)
        exec_cost += cost
        spread_cost += spread
        impact_cost += impact
        total_abs_hedge += abs(close_delta)
        hedge_events.append({
            'bar_time': row.bar_time,
            'event_type': 'close_flatten',
            'band': band,
            'method': method,
            'horizon_minutes': horizon_minutes,
            'hedge_delta': close_delta,
            'book_delta_after': 0.0,
            'exec_cost': cost,
            'spread_cost': spread,
            'impact_cost': impact,
        })

    result = {
        'trade_date': day_grid['trade_date'].iloc[0],
        'strategy': 'static_band',
        'band': band,
        'method': method,
        'horizon_minutes': horizon_minutes,
        'execution_cost': exec_cost,
        'spread_cost': spread_cost,
        'impact_cost': impact_cost,
        'delta_risk': risk_cost,
        'combined_loss': exec_cost + LAMBDA_RISK * risk_cost,
        'n_triggers': triggers,
        'total_abs_hedge_delta': total_abs_hedge,
        'eod_delta_before_flatten': -close_delta,
    }
    return result, pd.DataFrame(hedge_events)

## 8. Benchmark Frameworks

In [ ]:
def simulate_zero_hedge_day(day_grid: pd.DataFrame) -> dict:
    book_delta = 0.0
    risk_cost = 0.0
    day_grid = day_grid.reset_index(drop=True)
    for row in day_grid.itertuples(index=False):
        book_delta += float(row.option_delta_arrival)
        risk_cost += (book_delta ** 2) * (float(row.bvol_1m) ** 2) * BAR_YEAR_FRACTION
    close_row = day_grid.iloc[-1]
    close_delta = -book_delta
    exec_cost, spread_cost, impact_cost = estimate_child_order_cost(close_delta, close_row.es1_price, close_row.es1_adv_20d)
    return {
        'trade_date': day_grid['trade_date'].iloc[0],
        'strategy': 'zero_hedge',
        'band': np.nan,
        'method': 'close_only',
        'horizon_minutes': 0,
        'execution_cost': exec_cost,
        'spread_cost': spread_cost,
        'impact_cost': impact_cost,
        'delta_risk': risk_cost,
        'combined_loss': exec_cost + LAMBDA_RISK * risk_cost,
        'n_triggers': 0,
        'total_abs_hedge_delta': abs(close_delta),
        'eod_delta_before_flatten': -close_delta,
    }


def simulate_full_hedge_day(day_grid: pd.DataFrame) -> dict:
    exec_cost = 0.0
    spread_cost = 0.0
    impact_cost = 0.0
    total_abs_hedge = 0.0
    n_triggers = 0
    day_grid = day_grid.reset_index(drop=True)
    for row in day_grid.itertuples(index=False):
        arrival = float(row.option_delta_arrival)
        if arrival != 0.0:
            hedge_delta = -arrival
            cost, spread, impact = estimate_child_order_cost(hedge_delta, row.es1_price, row.es1_adv_20d)
            exec_cost += cost
            spread_cost += spread
            impact_cost += impact
            total_abs_hedge += abs(hedge_delta)
            n_triggers += 1
    return {
        'trade_date': day_grid['trade_date'].iloc[0],
        'strategy': 'full_hedge',
        'band': 0.0,
        'method': 'immediate',
        'horizon_minutes': 0,
        'execution_cost': exec_cost,
        'spread_cost': spread_cost,
        'impact_cost': impact_cost,
        'delta_risk': 0.0,
        'combined_loss': exec_cost,
        'n_triggers': n_triggers,
        'total_abs_hedge_delta': total_abs_hedge,
        'eod_delta_before_flatten': 0.0,
    }

## 9. Run the Baseline Grid

In [ ]:
results = []
hedge_logs = []

for trade_date, day_grid in bar_grid.groupby('trade_date'):
    day_grid = day_grid.sort_values('bar_time').reset_index(drop=True)
    results.append(simulate_zero_hedge_day(day_grid))
    results.append(simulate_full_hedge_day(day_grid))

    for band in BAND_GRID:
        for method in EXECUTION_METHODS:
            for horizon in HORIZON_GRID_MINUTES:
                day_result, day_hedges = simulate_static_band_day(day_grid, band, method, horizon)
                results.append(day_result)
                if not day_hedges.empty:
                    day_hedges['trade_date'] = trade_date
                    hedge_logs.append(day_hedges)

results_df = pd.DataFrame(results)
hedge_events = pd.concat(hedge_logs, ignore_index=True) if hedge_logs else pd.DataFrame()

results_df.to_csv(OUTPUT_RESULTS, index=False)
hedge_events.to_csv(OUTPUT_HEDGE_EVENTS, index=False)

print(f'Wrote results to {OUTPUT_RESULTS.relative_to(PROJECT_ROOT)}')
print(f'Wrote hedge events to {OUTPUT_HEDGE_EVENTS.relative_to(PROJECT_ROOT)}')
results_df.head()

## 10. Summarize Strategy Performance

In [ ]:
summary = (
    results_df.groupby(['strategy', 'band', 'method', 'horizon_minutes'], dropna=False)
    .agg(
        execution_cost=('execution_cost', 'sum'),
        delta_risk=('delta_risk', 'sum'),
        combined_loss=('combined_loss', 'sum'),
        n_triggers=('n_triggers', 'sum'),
        total_abs_hedge_delta=('total_abs_hedge_delta', 'sum'),
        avg_eod_delta_before_flatten=('eod_delta_before_flatten', 'mean'),
    )
    .reset_index()
    .sort_values('combined_loss')
)

summary.head(15)

## 11. Next Notebook Handoff

Notebook 03 should consume `baseline_strategy_results.csv` and build:

- efficient-frontier plots in execution-cost vs delta-risk space;
- benchmark comparisons versus zero hedge and full hedge;
- sensitivity tables for band width, execution horizon, impact parameters, and risk penalty;
- the final recommendation for the baseline static-band strategy.